Proceso de flujo contínuo

Frecuencia de muestreo de 1 Hz

Máquinas 1, 2 y 3 funcionan en paralelo. Sus salidas se combinan en un flujo

La salida del combinador se mide en 15 lugares distintos. Son los primeros parámetros a medir.

La salida entra en la máquina 4, y la 5 está a continuación.



In [ ]:
import pandas as pd
import numpy as np


In [ ]:
filepath = '../Datasets/continuous_factory_process.csv'

TARGET = "Stage1.Output.Measurement8.U.Actual"

df = pd.read_csv(filepath)
df.head(-5)

df['time_stamp'] = pd.to_datetime(df['time_stamp'])
df = df.set_index('time_stamp').sort_index()
df = df[~df.index.duplicated(keep='first')]
df = df.asfreq('1s')


duración del proceso = 3 h 55 min aprox

variables por columna:
0	0	Time stamp

1	2	Factory ambient conditions

3	6	First stage, Machine 1, raw material properties (material going in to Machine 1)

7	14	First stage, Machine 1 process variables

15	18	First stage, Machine 2, raw material properties (material going in to Machine 2)

19	26	First stage, Machine 2 process variables

27	30	First stage, Machine 3, raw material properties (material going in to Machine 3)

31	38	First stage, Machine 3 process variables

39	41	Combiner stage process parameters. Here we combines the outputs from Machines 1, 2, and 3.

42	71	PRIMARY OUTPUT TO CONTROL: Measurements of 15 features (in mm), along with setpoint or target for each

72	78	Second stage, Machine 4 process variables

79	85	Second stage, Machine 5 process variables

86	115	SECONDARY OUTPUT TO CONTROL: Measurements of 15 features (in mm), along with setpoint or target for each

---------------------------------------

delay entre la entrada de material a las máquinas y la salida; entrada del combinador a la salida del combinador; salida del combinador hasta la medida

se supone que variables que tienen muy poca variación (pasa con la temperatura) pueden ser rechazadas

In [ ]:
# Remove columns which contains "Stage2"
bad_words = ["Stage2", "Machine4", "Machine5", "Setpoint"]

df = df.drop(columns=[col for col in df.columns if any(word in col for word in bad_words)])
df = df.drop(columns=[col for col in df.columns if "Stage1.Output.Measurement" in col and col != TARGET])
print(df.columns.tolist())

Check missing values


In [ ]:
import pandas as pd

empty_list = []

for column in df.columns:
    is_null = df[column].isna()
    
    if not is_null.any():
        continue

    blocks = (~is_null).cumsum()
    null_dates = df[is_null].reset_index()
    time_col = null_dates.columns[0] 
    
    rangos = null_dates.groupby(blocks[is_null].values)[time_col].agg(
        Inicio='min',
        Fin='max'
    )
    
    rangos['Variable'] = column
    
    # 1. Calcular la duración y convertirla a segundos (número entero)
    rangos['Duracion_segundos'] = (rangos['Fin'] - rangos['Inicio']).dt.total_seconds().astype(int)
    
    # 2. Formatear las fechas para que siempre tengan la misma longitud y se alineen bien
    rangos['Inicio'] = rangos['Inicio'].dt.strftime('%Y-%m-%d %H:%M:%S')
    rangos['Fin'] = rangos['Fin'].dt.strftime('%Y-%m-%d %H:%M:%S')
    
    empty_list.append(rangos)

if empty_list:
    df_nulos = pd.concat(empty_list).reset_index(drop=True)
    df_nulos = df_nulos[['Variable', 'Inicio', 'Fin', 'Duracion_segundos']]
    
    # Imprimir forzando la alineación para que se lea perfecto en la consola
    print(df_nulos.to_string(justify='center'))
else:
    df_nulos = pd.DataFrame()
    print("No hay valores nulos en el DataFrame.")

Fill empty values using the mean

In [ ]:
for column in df.columns:
    if df[column].isna().any():
        df[column] = df[column].interpolate(method='linear', limit_direction='both')

# Check if still there are any null values after interpolation
for column in df.columns:
    if df[column].isna().any():
        print(f"Warning: Column '{column}' still has null values after interpolation.")

Statistical analysis

In [ ]:
# 1. Seleccionar solo las columnas numéricas (evita errores con fechas o texto)
df_num = df.select_dtypes(include=['number'])

# 2. Generar las estadísticas básicas y transponer (.T) para que las variables sean filas
stats = df_num.describe().T

# 3. Calcular el Coeficiente de Variación (CV = std / mean)
# Lo multiplicamos por 100 para expresarlo como porcentaje
stats['CV (%)'] = (stats['std'] / stats['mean']) * 100

# 4. (Opcional) Limpiar el formato para que sea más fácil de leer
# Redondeamos a 2 decimales para no tener números gigantes en pantalla
stats = stats.round(2)

# Mostrar la tabla final
print(stats.to_string())

In [ ]:
# Export dataset to csv. Insert the time_stamp column at the left and then export to csv. Then set again the time_stamp column as index
df.insert(0, "time_stamp", df.index)
df.to_csv("continuous_process_filtered.csv", index=False)
df.set_index("time_stamp", inplace=True)

# Observe data distribution with histograms (tsbox)
Máquina 1
- La distribución del FeederParameter de la máquina 1 tiene una única distribución parece que bastante expandida por ruido. Habría que hacer un filtrado para achatar la gausiana
Las señales 
- No vamos a usar la  

# Drop signals whose CV is under 1

In [ ]:
# 1. Seleccionar solo columnas numéricas
df_num = df.select_dtypes(include=[np.number])

# 2. Calcular el Coeficiente de Variación (CV = std / mean)
# Se usa np.abs por si la media fuera negativa
cv_series = (df_num.std() / df_num.mean()).abs()

# 3. Filtrar columnas donde CV < 1
columnas_cv_menor_1 = cv_series[cv_series < 1]

# 4. Mostrar el DataFrame resultante solo con esas columnas
df_filtrado = df[columnas_cv_menor_1.index]

print("--- Columnas con CV < 1 y su valor ---")
print(columnas_cv_menor_1)

print("\n--- Visualización de las columnas filtradas ---")
print(df_filtrado.head())

Análisis de señales gráfico

In [ ]:
import matplotlib.pyplot as plt

# 1. ESCRIBE AQUÍ LAS SEÑALES QUE QUIERES VER (Deben existir en tu DataFrame 'df')
# Ejemplo: señales_a_pintar = ["Stage2_Temp", "Machine4_Pressure"]
señales_a_pintar = df.columns.tolist()  # Esto pintará todas las señales, puedes cambiarlo a una lista específica 

# Comprobar que hemos puesto alguna señal
if not señales_a_pintar:
    print("Por favor, añade al menos una señal a la lista 'señales_a_pintar'.")
else:
    # 2. Crear las subfiguras (una debajo de otra)
    # Ajustamos la altura (2.5) multiplicada por el número de señales
    fig, axes = plt.subplots(
        nrows=len(señales_a_pintar), 
        ncols=1, 
        figsize=(12, 2.5 * len(señales_a_pintar)), 
        sharex=True
    )
    
    # Ajuste por si solo pones 1 señal en la lista (para que no falle el bucle)
    if len(señales_a_pintar) == 1:
        axes = [axes]
        
    # 3. Pintar cada señal en su gráfico correspondiente
    for ax, columna in zip(axes, señales_a_pintar):
        # Verifica que la columna exista para evitar errores tontos
        if columna in df.columns:
            ax.plot(df.index, df[columna], color='#1f77b4', linewidth=1.5)
            ax.set_title(columna, fontsize=12, fontweight='bold', loc='left')
            ax.grid(True, linestyle='--', alpha=0.7)
            # Ocultar el marco superior y derecho para que quede más limpio
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
        else:
            ax.text(0.5, 0.5, f"La columna '{columna}' no existe", 
                    ha='center', va='center', color='red')
            
    # Ajustar espacios para que no se superpongan los títulos con los ejes
    plt.tight_layout()
    plt.show()

# TODO after visualization
- Analizar estadísticamente desde el inicio de la señal hasta la hora 11:11. Hacer otro análisis aparte desde las 11:11 hasta el final
- Machine2.MotorRPM.C.Actual no aporta
- Machine2.Zone1Temperature parece que tampoco
- Machine2.Zone2Temperature parece que tampoco
- Machine2.ExitZoneTemperature mantiene la media, pero la varianza cambia con el paso del tiempo

- Media movil 
-- Machine2.MaterialPressure
-- Machine2.MotorAmperage

Training split

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

target_col = TARGET
feature_cols = [c for c in df.columns if c not in ['time_stamp', target_col]]

X_raw = df[feature_cols].values
y_raw = df[[target_col]].values

# 2. Split cronológico (70% Train, 15% Val, 15% Test)
n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train_raw, y_train_raw = X_raw[:train_end], y_raw[:train_end]
X_val_raw, y_val_raw     = X_raw[train_end:val_end], y_raw[train_end:val_end]
X_test_raw, y_test_raw   = X_raw[val_end:], y_raw[val_end:]

# 3. Escalado independiente (Fit solo en Train)
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train = scaler_X.fit_transform(X_train_raw)
y_train = scaler_y.fit_transform(y_train_raw)

X_val = scaler_X.transform(X_val_raw)
y_val = scaler_y.transform(y_val_raw)

X_test = scaler_X.transform(X_test_raw)
y_test = scaler_y.transform(y_test_raw)